# S8 · Lab 02 — La Trampa del Feature Engineering (el “éxito” peligroso)

## Narrativa
El éxito del Lab 01 te puede dar una idea peligrosa:

> “Si grado 3 ayuda, ¿por qué no subimos a grado 10 y que el ordenador haga el trabajo?”

En este laboratorio vas a comprobar por qué esa intuición **se descontrola**:
- el número de variables explota,
- la interpretabilidad operativa colapsa,
- y aparece una “caja negra” hecha por el propio ingeniero.

---

## Qué vas a producir
1) Un gráfico que muestre cómo crece el número de columnas según el grado.  
2) Un “buscador de agujas”: localizar una feature por índice y “traducirla” a negocio.  
3) Una reflexión profesional: *¿cómo justificas una decisión si no entiendes tus variables?*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# 1) Dataset con 20 variables (simula un caso real: muchas señales)
X, y = make_classification(
    n_samples=800,
    n_features=20,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

print("Shape original:", X.shape)

In [ ]:
# 2) Explosión visual: cuántas columnas aparecen por grado
degrees = [1, 2, 3]
n_cols = []

for d in degrees:
    poly = PolynomialFeatures(degree=d, include_bias=False)
    Xp = poly.fit_transform(X)
    n_cols.append(Xp.shape[1])

plt.figure()
plt.bar([str(d) for d in degrees], n_cols)
plt.title("Explosión de columnas según el grado (20 variables de entrada)")
plt.xlabel("Grado polinómico")
plt.ylabel("Número de columnas resultantes")
plt.show()

list(zip(degrees, n_cols))

## Paso 3 — “Buscador de agujas” (pérdida de interpretabilidad)

Ahora vamos a hacer algo deliberadamente incómodo:

1) Genera las features de grado 3.  
2) Elige una columna “lejana” (por ejemplo, 800 o 1250).  
3) Intenta darle un nombre “de negocio”.

**Ejemplo de traducción absurda (solo para inspirarte):**  
“edad² · código_postal³ · saldo”  

La pregunta no es si puedes decirlo.  
La pregunta es si **eso es una explicación profesional**.


In [ ]:
# 3) Generamos grado 3
poly3 = PolynomialFeatures(degree=3, include_bias=False)
X3 = poly3.fit_transform(X)

feature_names = poly3.get_feature_names_out([f"x{i}" for i in range(X.shape[1])])

print("Shape tras degree=3:", X3.shape)
print("Ejemplo de 5 features:", feature_names[:5])

## Extra visual (para entender *qué* ha explotado)
No solo han aumentado las columnas: también han aumentado las **interacciones** y los **grados** mezclados.

Vamos a:
- calcular el **grado total** de cada feature (suma de exponentes),
- contar cuántas variables originales aparecen en cada término,
- y visualizar la distribución.


In [ ]:
import re

def feature_degree_and_terms(name: str):
    # name like "x3^2 x11 x7^3" (scikit uses spaces)
    parts = name.split()
    deg = 0
    terms = 0
    for p in parts:
        m = re.match(r"(x\d+)(\^(\d+))?$", p)
        if not m:
            continue
        terms += 1
        exp = int(m.group(3)) if m.group(3) else 1
        deg += exp
    return deg, terms

degrees = np.zeros(len(feature_names), dtype=int)
terms = np.zeros(len(feature_names), dtype=int)

for i, fn in enumerate(feature_names):
    d, t = feature_degree_and_terms(fn)
    degrees[i] = d
    terms[i] = t

print("Grado total (min/max):", degrees.min(), degrees.max())
print("Nº términos (min/max):", terms.min(), terms.max())

In [ ]:
# Distribución visual
plt.figure()
plt.hist(degrees, bins=range(degrees.min(), degrees.max()+2))
plt.title("Distribución de grados totales (degree=3)")
plt.xlabel("Grado total de la feature")
plt.ylabel("Número de features")
plt.show()

plt.figure()
plt.hist(terms, bins=range(terms.min(), terms.max()+2))
plt.title("Cuántas variables se multiplican en cada feature (nº de términos)")
plt.xlabel("Número de variables involucradas (términos)")
plt.ylabel("Número de features")
plt.show()

In [ ]:
# Top-10 features "más complejas" (más términos y grado alto)
complex_score = terms * 10 + degrees  # prioriza términos, luego grado
top = np.argsort(complex_score)[::-1][:10]

print("Top-10 features más complejas (por nº de términos y grado):")
for i in top:
    print(f"{i:5d} | grado={degrees[i]} | términos={terms[i]} | {feature_names[i]}")

In [ ]:
# 4) Elige una "aguja" por índice (puedes cambiarlo)
s = input("Elige un índice (ENTER para 1250): ").strip()
idx = 1250 if s == "" else int(s)
idx = max(0, min(idx, len(feature_names)-1))

print("Índice elegido:", idx)
print("Nombre simbólico:", feature_names[idx])

# Contexto alrededor
start = max(0, idx-3); end = min(len(feature_names), idx+4)
print("\nAlrededor del índice:")
for i in range(start, end):
    print(i, "->", feature_names[i])

# Información estructural de la aguja
d, t = feature_degree_and_terms(feature_names[idx])
print(f"\nEstructura: grado_total={d}, términos={t}")

### Reto (obligatorio)
1) Escribe una frase intentando explicar qué significa esa variable en un contexto de negocio (banco, marketing, salud…).
2) Luego responde: **¿confiarías tu trabajo a una explicación así?** ¿Por qué?

---

### Mini-escenario profesional (para forzar criterio)
Imagina que eres analista en un banco. Tu jefe te pregunta:

> “¿Por qué hemos rechazado este préstamo?”

Si tu respuesta es:

> “Porque `x3^2 · x11 · x7^3` era negativo”

¿qué crees que pasará con tu carrera?

No contestes con drama. Contesta con criterio:
- explicabilidad,
- auditoría,
- confianza,
- depuración,
- responsabilidad.


## Preguntas de cierre (responde aquí)

1) ¿Qué aprendiste del gráfico de crecimiento por grado?  
2) ¿En qué punto deja de ser “ingeniería” y pasa a ser “ruido combinatorio”?  
3) ¿Qué problema profesional crea un modelo que depende de features que no puedes explicar?  
4) Conecta con Lab 01: ¿por qué “más grado” no es una estrategia universal?

